# RF retinotopy per session

For every treadmill/motor session (the same set used in `decoder_treadmill_cut` /
`submit_rf_only.py`), this notebook:

1. Counts the **number of significant receptive fields** (contra RF peak > `N_STD`
   SD above the ipsilateral RF, via `find_sig_rfs`, default `n_std=6`).
2. Builds the **average RF image** across significant cells (peak-normalised, depth
   collapsed) — a 16×24 elevation×azimuth map.
3. Finds the **retinotopic center** of each session (peak of the average image and,
   as a cross-check, the median of per-neuron RF-peak locations), in azimuth /
   elevation degrees.
4. Plots a **retinotopy map** showing where each recording sampled visual space.

Run this once all the RF fits (`neurons_df_rf_only_treadmill.pickle`) are done.
Sessions whose pickle is missing are listed and skipped.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import flexiznam as flz
from cottage_analysis.summary_analysis import get_session_list
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.analysis.spheres.rf_fitting import find_sig_rfs

# ---- config -----------------------------------------------------------------
PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
SESSIONS_TO_EXCLUDE = ["PZAG22.1b_S20260220"]
RF_PICKLE = "neurons_df_rf_only_treadmill.pickle"
SUFFIX = "_closedloop_treadmill"        # RF fit condition to analyse

N_STD = 6                                # significance threshold (find_sig_rfs)
N_ELE, N_AZI = 16, 24                    # elevation x azimuth pixels (contra hemifield)
RESOLUTION = 5.0                         # degrees per pixel

# pixel -> degree mapping (matches rf_analysis.find_rf_centers / plot_rf extent)
#   azimuth : (col + 0.5) * RESOLUTION            -> 2.5 .. 117.5 deg  (0-120)
#   elevation: (row + 0.5 - N_ELE/2) * RESOLUTION -> -37.5 .. 37.5 deg (-40-40)
AZI_EXTENT = (0.0, N_AZI * RESOLUTION)
ELE_EXTENT = (-N_ELE / 2 * RESOLUTION, N_ELE / 2 * RESOLUTION)


def pixel_to_deg(row, col):
    azi = (col + 0.5) * RESOLUTION
    ele = (row + 0.5 - N_ELE / 2) * RESOLUTION
    return azi, ele

In [ ]:
# ---- session list (same query as submit_rf_only.py) -------------------------
sessions = []
for project in PROJECTS:
    flm = flz.get_flexilims_session(project_id=project)
    for s in get_session_list.get_motor_session_list(
        flm, exclude_sessions=SESSIONS_TO_EXCLUDE
    ):
        sessions.append((project, s))
print(f"{len(sessions)} motor sessions across {len(PROJECTS)} projects")

In [ ]:
def load_rf_neurons_df(project, session_name):
    """Load the RF-only neurons_df for a session, or None if not fitted yet."""
    flm = flz.get_flexilims_session(project_id=project)
    neurons_ds = pipeline_utils.create_neurons_ds(
        session_name=session_name, flexilims_session=flm,
        project=project, conflicts="skip",
    )
    path = neurons_ds.path_full.parent / RF_PICKLE
    return pd.read_pickle(path) if path.exists() else None


def session_rf_summary(df):
    """Compute significance, average RF image and retinotopic center."""
    coef = np.stack(df[f"rf_coef{SUFFIX}"].values)          # (rois, folds, feat)
    coef_ipsi = np.stack(df[f"rf_coef_ipsi{SUFFIX}"].values)
    n_rois, n_folds, n_feat = coef.shape
    ndepths = (n_feat - 1) // (N_ELE * N_AZI)

    # significance: find_sig_rfs expects (folds, features, rois)
    sig, sig_ipsi = find_sig_rfs(
        coef.transpose(1, 2, 0), coef_ipsi.transpose(1, 2, 0), n_std=N_STD
    )

    # fold-averaged, bias-dropped RF volume -> (rois, depth, ele, azi)
    vol = np.nanmean(
        coef[:, :, :-1].reshape(n_rois, n_folds, ndepths, N_ELE, N_AZI), axis=1
    )
    rf2d = np.nanmax(vol, axis=1)                            # collapse depth -> (rois, ele, azi)

    # per-neuron center = 3D peak of the RF volume (only for significant, non-nan)
    centers = np.full((n_rois, 2), np.nan)                   # columns: azi, ele
    for i in range(n_rois):
        if sig[i] and not np.all(np.isnan(vol[i])):
            _, r, c = np.unravel_index(np.nanargmax(vol[i]), vol[i].shape)
            centers[i] = pixel_to_deg(r, c)

    # session average image over significant neurons (peak-normalised)
    avg_img = np.full((N_ELE, N_AZI), np.nan)
    if sig.sum() > 0:
        imgs = rf2d[sig].astype(float).copy()
        peak = np.nanmax(imgs.reshape(imgs.shape[0], -1), axis=1)
        peak[~(peak > 0)] = np.nan
        imgs = imgs / peak[:, None, None]
        avg_img = np.nanmean(imgs, axis=0)

    # session center from the average image peak, plus median of neuron centers
    if np.all(np.isnan(avg_img)):
        center_img = (np.nan, np.nan)
    else:
        r, c = np.unravel_index(np.nanargmax(avg_img), avg_img.shape)
        center_img = pixel_to_deg(r, c)
    center_med = (
        (np.nanmedian(centers[sig, 0]), np.nanmedian(centers[sig, 1]))
        if sig.sum() else (np.nan, np.nan)
    )

    return dict(
        n_rois=n_rois, n_sig=int(sig.sum()), ndepths=ndepths,
        sig=sig, avg_img=avg_img, centers=centers,
        center_img=center_img, center_med=center_med,
    )

In [ ]:
# ---- compute for every session ---------------------------------------------
results, missing = {}, []
for project, s in sessions:
    df = load_rf_neurons_df(project, s)
    if df is None:
        missing.append((project, s))
        continue
    try:
        results[(project, s)] = session_rf_summary(df)
    except Exception as e:
        print(f"ERROR {s}: {e}")

print(f"summarised {len(results)} sessions; {len(missing)} missing pickle")
for p, s in missing:
    print("  missing:", p, s)

In [ ]:
# ---- summary table ----------------------------------------------------------
summary = pd.DataFrame([
    dict(
        project=p, session=s, mouse=s.split("_")[0],
        n_rois=r["n_rois"], n_sig=r["n_sig"],
        frac_sig=(r["n_sig"] / r["n_rois"] if r["n_rois"] else np.nan),
        center_azi=r["center_img"][0], center_ele=r["center_img"][1],
        center_azi_med=r["center_med"][0], center_ele_med=r["center_med"][1],
    )
    for (p, s), r in results.items()
]).sort_values("session").reset_index(drop=True)
summary

In [ ]:
# ---- save the summary table for reuse in other notebooks --------------------
from pathlib import Path
from flexiznam.config import PARAMETERS

# Save in the ccyp_l5_3d_vision project folder by default.
SAVE_PROJECT = "ccyp_l5_3d_vision"
PROCESSED_ROOT = Path(PARAMETERS["data_root"]["processed"])
SUMMARY_PATH = PROCESSED_ROOT / SAVE_PROJECT / "rf_retinotopy_summary.parquet"

summary.to_parquet(SUMMARY_PATH)
print("saved", SUMMARY_PATH)

# Reload it from another notebook with:
#     import pandas as pd
#     from flexiznam.config import PARAMETERS
#     from pathlib import Path
#     summary = pd.read_parquet(
#         Path(PARAMETERS["data_root"]["processed"])
#         / "ccyp_l5_3d_vision" / "rf_retinotopy_summary.parquet"
#     )
# (use .to_csv / pd.read_csv instead if you want a human-readable copy)

## 1. Number of significant RFs per session

In [ ]:
s2 = summary.sort_values("n_sig", ascending=False).reset_index(drop=True)
colors = {"ccyp_l5_3d_vision": "C0", "colasa_3d-vision_revisions": "C1"}
fig, ax = plt.subplots(figsize=(11, max(4, 0.32 * len(s2))))
ax.barh(s2.session, s2.n_sig, color=[colors[p] for p in s2.project])
ax.invert_yaxis()
ax.set_xlabel("# significant RFs")
ax.set_title(f"Significant RFs per session (find_sig_rfs, n_std={N_STD})")
for y, (n, f) in enumerate(zip(s2.n_sig, s2.frac_sig)):
    ax.text(n + max(s2.n_sig) * 0.01, y, f"{n}  ({f*100:.0f}%)",
            va="center", fontsize=7)
handles = [plt.Line2D([0], [0], marker="s", ls="", color=c, label=p)
           for p, c in colors.items()]
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.show()

## 2. Average RF image per session

Peak-normalised average over significant cells, collapsed across depth. The cyan
`+` marks the retinotopic center (peak of the average image). Axes are azimuth
(0–120°) × elevation (−40–40°).

In [ ]:
keys = list(results.keys())
ncol = 6
nrow = int(np.ceil(len(keys) / ncol)) if keys else 1
fig, axes = plt.subplots(nrow, ncol, figsize=(2.8 * ncol, 2.4 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax, k in zip(axes, keys):
    r = results[k]
    ax.imshow(r["avg_img"], origin="lower", cmap="magma", aspect="auto",
              extent=[AZI_EXTENT[0], AZI_EXTENT[1], ELE_EXTENT[0], ELE_EXTENT[1]])
    az, el = r["center_img"]
    if not np.isnan(az):
        ax.plot(az, el, "c+", ms=11, mew=2)
    ax.set_title(f"{k[1]}\nn_sig={r['n_sig']}  ({az:.0f}, {el:.0f})°", fontsize=7)
    ax.set_xticks([0, 60, 120]); ax.set_yticks([-40, 0, 40])
    ax.tick_params(labelsize=6)
for ax in axes[len(keys):]:
    ax.axis("off")
fig.supxlabel("Azimuth (deg)"); fig.supylabel("Elevation (deg)")
plt.tight_layout()
plt.show()

## 3. Retinotopic location of each recording

Each point is a session, placed at its average-image center in visual space,
coloured by mouse. This shows which retinotopic location each recording sampled.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
mice = sorted(summary.mouse.unique())
mcol = dict(zip(mice, plt.cm.tab20(np.linspace(0, 1, max(len(mice), 1)))))
for _, row in summary.iterrows():
    if np.isnan(row.center_azi):
        continue
    ax.scatter(row.center_azi, row.center_ele, color=mcol[row.mouse],
               s=70, edgecolor="k", zorder=3)
    ax.annotate(row.session.split("_S")[-1], (row.center_azi, row.center_ele),
                fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.set_xlim(0, 120); ax.set_ylim(-40, 40)
ax.set_xlabel("Azimuth (deg)"); ax.set_ylabel("Elevation (deg)")
ax.set_title("Retinotopic center of each recording")
ax.grid(alpha=0.3)
handles = [plt.Line2D([0], [0], marker="o", ls="", mfc=mcol[m], mec="k", label=m)
           for m in mice]
ax.legend(handles=handles, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Notes
- Significance uses `find_sig_rfs` (contra vs ipsi, `n_std=6`) over **all fitted
  ROIs** in the RF-only pickle. That pickle has no `iscell` / depth-tuning columns,
  so — unlike `load_sig_rf` — no additional cell/tuning filter is applied. Lower
  `N_STD` for a more permissive count.
- `center_azi/ele` come from the peak of the session average image;
  `center_azi_med/ele_med` are the median of per-neuron RF peaks (cross-check).
- The average image collapses depth with a max; switch `np.nanmax` to `np.nanmean`
  in `session_rf_summary` if you prefer a mean-over-depth map.